# **HELWAN: STREET-BUILDING INTERFACE ANALYSIS**

## **Importing data from GeoPackages**

In [ ]:
# Importing python libraries 
import geopandas as gpd
import pandas as pd
import numpy as np
import fiona
import matplotlib.pyplot as plt
import warnings

In [ ]:
import os

DATA_DIR = r"C:\Users\alexa\Urban morphometrics\notebooks\Street-Building Interface\Data\Helwan"

streets    = gpd.read_file(os.path.join(DATA_DIR, "helwan_roads.gpkg"))
bld_res    = gpd.read_file(os.path.join(DATA_DIR, "residential_buildings.gpkg"))
bld_nonres = gpd.read_file(os.path.join(DATA_DIR, "non_residential_buildings.gpkg"))
bld_mix    = gpd.read_file(os.path.join(DATA_DIR, "mixed_use_buildings.gpkg"))

In [ ]:
print(f"streets:    {len(streets):6d} features  | CRS: {streets.crs}")
print(f"residential:{len(bld_res):6d} features  | CRS: {bld_res.crs}")
print(f"non_res:    {len(bld_nonres):6d} features  | CRS: {bld_nonres.crs}")
print(f"mix_use:    {len(bld_mix):6d} features  | CRS: {bld_mix.crs}")

In [ ]:
CRS_PROJ = "EPSG:32636"

print("STREETS")
print(streets.columns.tolist())
print(f"Geometria: {streets.geometry.geom_type.value_counts().to_dict()}")

print("\n RESIDENTIAL")
print(bld_res.columns.tolist())

print("\n NON RESIDENTIAL")
print(bld_nonres.columns.tolist())

print("\n MIX USE")
print(bld_mix.columns.tolist())

We detect some issues: 
1. Streets are MultiLineStrings. Momepystreetscape works with LineStrings. Therefore we need to explode them
2. We know that theres a Heights column in every geopackage. But we need to check how Heights are distributued in the 3 layers

In [ ]:
streets = streets.explode(index_parts=False).reset_index(drop=True)
print(f"Streets exploded: {len(streets)} | {streets.geometry.geom_type.value_counts().to_dict()}")

In [ ]:
for name, gdf in [("residential", bld_res), ("non_res", bld_nonres), ("mix_use", bld_mix)]:
    print(f"\n{name} — Heights:")
    print(gdf["Heights"].describe())
    print(f"  Valores únicos: {sorted(gdf['Heights'].dropna().unique())[:15]}")
    print(f"  Nulos: {gdf['Heights'].isna().sum()}")

Seems like the heights are in Floors and not meters. Therefore we need to convert to meters. We considered 3.2m for each floor

In [ ]:
HEIGHT_PPF = 3.2  # meters for floor

def clean_height(val, ppf=HEIGHT_PPF):
    s = str(val).strip()
    if "اكثر" in s or "أكثر" in s:
        return 13 * ppf  # "more than 12" we assume 13 floors (we will cap at 13 floors)
    try:
        return float(s) * ppf
    except:
        return np.nan

bld_res["height_m"]    = bld_res["Heights"].apply(clean_height)
bld_nonres["height_m"] = bld_nonres["Heights"].apply(clean_height)
bld_mix["height_m"]    = bld_mix["Heights"].apply(clean_height)

for name, gdf in [("residential", bld_res), ("non_res", bld_nonres), ("mix_use", bld_mix)]:
    print(f"{name}: min={gdf['height_m'].min():.1f}m  "
          f"max={gdf['height_m'].max():.1f}m  "
          f"média={gdf['height_m'].mean():.1f}m")

Now lets check the geometry of the buildings, if they are polygons or multipolygons. Streets were multilinestrings, so probably the buildings are also a multi-:

In [ ]:
print(bld_res.geometry.geom_type.value_counts())
print(bld_nonres.geometry.geom_type.value_counts())
print(bld_mix.geometry.geom_type.value_counts())

The 3 layers are all MultiPolygons. We need to explode them also:

In [ ]:
bld_res    = bld_res.explode(index_parts=False).reset_index(drop=True)
bld_nonres = bld_nonres.explode(index_parts=False).reset_index(drop=True)
bld_mix    = bld_mix.explode(index_parts=False).reset_index(drop=True)

print(f"residential: {len(bld_res)} | {bld_res.geometry.geom_type.value_counts().to_dict()}")
print(f"non_res:     {len(bld_nonres)} | {bld_nonres.geometry.geom_type.value_counts().to_dict()}")
print(f"mix_use:     {len(bld_mix)} | {bld_mix.geometry.geom_type.value_counts().to_dict()}")

Now we need to combine the 3 layers into 1 layer to run Streetscape momepy. Since it reads only 1 GeoDataFRame of buildings to calculate the morphological descriptors (open space, setback, etc)
Nevertheless we keep the categories saved, under category, to distinguish the types of buildings.
So, lets combine them:

In [ ]:
bld_res["category"]    = "residential"
bld_nonres["category"] = "non_residential"
bld_mix["category"]    = "mix_use"

buildings = pd.concat([
    bld_res[["geometry", "height_m", "category"]],
    bld_nonres[["geometry", "height_m", "category"]],
    bld_mix[["geometry", "height_m", "category"]]
]).reset_index(drop=True)

buildings = gpd.GeoDataFrame(buildings, crs=CRS_PROJ)

print(f"Total edifícios: {len(buildings)}")
print(buildings["category"].value_counts())
print(f"Geometrias inválidas: {(~buildings.geometry.is_valid).sum()}")

We can also check that there was a invalid geometry. We should remove it before advancing

In [ ]:
buildings = buildings[buildings.geometry.is_valid].copy()
buildings = buildings.reset_index(drop=True)
print(f"Edifícios após limpeza: {len(buildings)}")

Lets make a static map to check the streets:

In [ ]:
# Quick static map to inspect streets and buildings

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 10))

# Buildings first, so streets appear on top
buildings.plot(
    ax=ax,
    column="category",
    categorical=True,
    cmap="Set2",
    linewidth=0.2,
    edgecolor="grey",
    alpha=0.6,
    legend=True
)

# Streets on top, coloured by road width
streets.plot(
    ax=ax,
    column="Road width",
    cmap="Blues",
    linewidth=0.8,
    alpha=0.9,
    legend=True
)

ax.set_title("Streets and buildings — Helwan", fontsize=14)
ax.axis("off")

plt.tight_layout()
plt.show()

We can see that there still are some buildings without a street nearby, mostly non_residental (green colour) buildings. Its probably a bit hard to spot, but the groups of green buildings dont have streets. 
Streetscape momepy will not work properly in this situation because it needs buildings and streets to measure the morphology.
Also, this is a big area to analyse and will take 5h-6h just to run streetscape analysis.
So, lets define a smaller study area and in the meantime we can remove the areas where buildings dont have streets (large green areas in the map, probably industrial areas).
So, in Qgis we made a polygon manually selecting a study-area. Now lets import it to this notebook:

In [ ]:
from pathlib import Path
DATA_DIR = Path(DATA_DIR)

study_area = gpd.read_file(DATA_DIR / "Study_area_final.gpkg").to_crs(CRS_PROJ)

### Clip datasets to the study area

This step restricts the street network and building footprints to the selected study area.  
Only geometries that intersect the study area boundary are retained, creating two working subsets: `streets_study` and `buildings_study`.

The number of selected streets and buildings is then printed to check the size of the clipped datasets

In [ ]:
streets_study   = streets[streets.intersects(study_area.geometry[0])].copy()
buildings_study = buildings[buildings.intersects(study_area.geometry[0])].copy()

print(f"Streets:   {len(streets_study)}")
print(f"Buildings: {len(buildings_study)}")
print(buildings_study["category"].value_counts())

Looks like we have much less streets and buildings. Lets plot a explore map to check what we kept:

In [ ]:
m = streets_study.to_crs(epsg=4326).explore(
    color="black",
    tiles=None,
    style_kwds={"weight": 1.5, "opacity": 0.7},
    name="Streets"
)

colors = {"residential": "#d4d0c8", "non_residential": "#d7191c", "mix_use": "#fdae61"}
for cat, color in colors.items():
    buildings_study[buildings_study["category"] == cat].to_crs(epsg=4326).explore(
        color=color,
        style_kwds={"weight": 0.5, "fillOpacity": 0.6},
        m=m,
        name=cat
    )

import folium
folium.LayerControl().add_to(m)
m

Seems we got a good target area for the analysis. 

NOTE: Theres always the possibility to reduce the number of sightlines of the streetscape analysis (by increasing the lenght from 10m to 20 or 30m), with this we could reduce the time run the code. But the issue is that 20 or 30m, or even 15m, doenst work for Helwan. The city has buildings with about 11m or 12m width, so a 15, 20 or 30m sightline would not catch the diversity in the buildings. Check the streetscape momepy paper for further explanation of this. 
Therefore, we will run a 10m sightline analysis of this area. It will take a long time but the end results will probably reflect the diversity of street-building interfaces.
Lets save everything in a cache before keep going:

In [ ]:
streets_study.to_file(DATA_DIR / "helwan_cache.gpkg", layer="streets", driver="GPKG")
buildings_study.to_file(DATA_DIR / "helwan_cache.gpkg", layer="buildings", driver="GPKG")
study_area.to_file(DATA_DIR / "helwan_cache.gpkg", layer="study_area", driver="GPKG")
print("Cache guardado.")

In [ ]:
# Create or load a streets_clean layer for momepy.Streetscape

cache_path = DATA_DIR / "helwan_cache.gpkg"

try:
    streets_clean = gpd.read_file(cache_path, layer="streets_clean").to_crs(CRS_PROJ)
    print(f"streets_clean loaded from cache: {len(streets_clean)} segments")

except Exception:
    print("No cached streets_clean layer found. Creating one from streets_study...")

    import momepy

    streets_clean = streets_study.copy()
    streets_clean = streets_clean.explode(index_parts=False).reset_index(drop=True)
    streets_clean = streets_clean[
        streets_clean.geometry.notna()
        & ~streets_clean.geometry.is_empty
        & streets_clean.geometry.geom_type.isin(["LineString", "MultiLineString"])
    ].copy()

    # Remove / simplify interstitial false nodes when possible.
    # This is useful for Streetscape, but centrality is recalculated later
    # from a separate fully noded topology.
    try:
        streets_clean = momepy.remove_false_nodes(streets_clean)
    except Exception as err:
        print(f"momepy.remove_false_nodes failed; using exploded streets_study instead. Reason: {err}")

    streets_clean.to_file(cache_path, layer="streets_clean", driver="GPKG")
    print(f"streets_clean saved to cache: {len(streets_clean)} segments")


In [ ]:
import momepy 

streets_clean   = gpd.read_file(DATA_DIR / "helwan_cache.gpkg", layer="streets_clean").to_crs(CRS_PROJ)
buildings_study = gpd.read_file(DATA_DIR / "helwan_cache.gpkg", layer="buildings").to_crs(CRS_PROJ)

print(f"streets_clean: {len(streets_clean)} | buildings: {len(buildings_study)}")

print("A correr momepy.Streetscape...")
sc = momepy.Streetscape(
    streets_clean,
    buildings_study,
    height_col="height_m",
    sightline_length=50,
    tangent_length=300,
    sightline_spacing=10,
)
print("Concluído.")

In [ ]:
street_df = sc.street_level()
print(f"Segmentos: {len(street_df)}")
print(street_df[["os", "built_coverage", "sb", "HW"]].describe().round(2))

In [ ]:
street_df = sc.street_level()
street_df = street_df.reset_index(drop=True)
street_df["seg_id"]   = range(len(street_df))
street_df["length_m"] = street_df.geometry.length

street_df.to_file(DATA_DIR / "helwan_cache.gpkg", layer="streetscape", driver="GPKG")
print(f"Guardado. Segmentos: {len(street_df)}")

In [ ]:
VAR_OS = "os"
VAR_CR = "built_coverage"
VAR_SB = "sb"
VAR_HW = "HW"

for v in [VAR_OS, VAR_CR, VAR_SB, VAR_HW]:
    ok = v in street_df.columns and street_df[v].notna().sum() > 0
    print(f"  {v}: {'OK' if ok else 'ERRO'}")

In [ ]:
print(f"Comprimento médio dos segmentos:")
print(f"  Helwan:    {street_df['length_m'].mean():.1f}m")
print(f"  Mediana:   {street_df['length_m'].median():.1f}m")
print(f"  Max:       {street_df['length_m'].max():.1f}m")

## **Centrality analysis**

## **Centrality analysis — corrected topological workflow**

The previous centrality block was producing unreliable values because the graph was built from line endpoints only. The street axes were not fully noded, so geometric crossings were often not graph connections.

This corrected workflow calculates **edge betweenness centrality** on a fully noded topological version of the same geometry used downstream in `street_df`, then transfers the values back to the `street_df` segments used in the rest of the analysis.

Key changes:
1. Build centrality from `street_df`, not from a different street layer.
2. Node the linework at real intersections before graph construction.
3. Do not keep only the largest component by default.
4. Assign centrality back to the original streetscape segments by length-weighted spatial overlap.


In [ ]:
# Edge betweenness centrality for non-topological street axes

import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import networkx as nx

from shapely.geometry import LineString, MultiLineString, GeometryCollection
from shapely.ops import transform, snap as shapely_snap

# Shapely 2.x has these at top level. The fallback keeps this cell usable in older environments.
try:
    from shapely import union_all, node, get_parts, set_precision
except Exception:
    from shapely.ops import unary_union

    def union_all(geoms):
        return unary_union(list(geoms))

    def node(geom):
        # unary_union of linework usually fully nodes line intersections in GEOS
        return unary_union(geom)

    def get_parts(geom):
        if hasattr(geom, "geoms"):
            return list(geom.geoms)
        return [geom]

    def set_precision(geom, grid_size):
        return geom


# -----------------------------
# Parameters
# -----------------------------

# Use the same projected CRS already used in the notebook.
# Helwan is in UTM zone 36N, so EPSG:32636 is a reasonable metric fallback.
CENTRALITY_CRS = globals().get("CRS_PROJ", "EPSG:32636")

# Precision grid in metres. Keep this small: it is for numerical robustness, not cartographic generalisation.
CENTRALITY_PRECISION_GRID = 0.01

# Optional snap tolerance in metres.
# Start with 0.0. If diagnostics still show too many disconnected components,
# try 0.5 or 1.0, then inspect the result visually.
CENTRALITY_SNAP_TOLERANCE = 0.0

# Remove tiny sliver segments created by noding.
CENTRALITY_MIN_SEGMENT_LENGTH = 0.50

# Node key precision. With metric CRS, 3 decimals = millimetre-level keys.
CENTRALITY_NODE_DECIMALS = 3

# By default, calculate centrality on all components.
# Setting this to True reproduces the common "largest connected component only" approach,
# but it will assign zero centrality to all smaller components.
CENTRALITY_ONLY_LARGEST_COMPONENT = False

# Exact betweenness is safest. If the graph is too large/slow, set e.g. CENTRALITY_APPROX_K = 1000.
CENTRALITY_APPROX_K = None
CENTRALITY_RANDOM_SEED = 42


# -----------------------------
# Input layer
# -----------------------------

if "street_df" not in globals():
    raise NameError("street_df does not exist. Run the Streetscape cells before this centrality block.")

if "seg_id" not in street_df.columns:
    street_df = street_df.reset_index(drop=True)
    street_df["seg_id"] = np.arange(len(street_df), dtype=int)

centrality_source = street_df[["seg_id", "geometry"]].copy()

if centrality_source.crs is None:
    raise ValueError("street_df has no CRS. Define its CRS before calculating centrality.")

if centrality_source.crs.is_geographic:
    centrality_source = centrality_source.to_crs(CENTRALITY_CRS)

centrality_source = centrality_source.explode(index_parts=False).reset_index(drop=True)
centrality_source = centrality_source[
    centrality_source.geometry.notna()
    & ~centrality_source.geometry.is_empty
    & centrality_source.geometry.geom_type.isin(["LineString", "MultiLineString"])
].copy()

# Force 2D coordinates in case the source has Z values.
def _force_2d(geom):
    try:
        return transform(lambda x, y, z=None: (x, y), geom)
    except Exception:
        return geom

centrality_source["geometry"] = centrality_source.geometry.map(_force_2d)
centrality_source["geometry"] = centrality_source.geometry.map(
    lambda g: set_precision(g, CENTRALITY_PRECISION_GRID)
)

print(f"Centrality source segments: {len(centrality_source):,}")
print(f"CRS used for centrality: {centrality_source.crs}")
print(f"Total source length: {centrality_source.length.sum():,.1f} m")


In [ ]:
# Build a fully noded topological segment layer

linework = union_all(centrality_source.geometry.to_numpy())

if CENTRALITY_SNAP_TOLERANCE and CENTRALITY_SNAP_TOLERANCE > 0:
    print(f"Applying self-snap tolerance: {CENTRALITY_SNAP_TOLERANCE} m")
    linework = shapely_snap(linework, linework, CENTRALITY_SNAP_TOLERANCE)

noded_linework = node(linework)

topological_parts = []
for part in get_parts(noded_linework):
    if part.is_empty:
        continue

    if part.geom_type == "LineString":
        topological_parts.append(part)
    elif part.geom_type == "MultiLineString":
        topological_parts.extend([g for g in part.geoms if not g.is_empty])
    elif part.geom_type == "GeometryCollection":
        topological_parts.extend(
            [g for g in part.geoms if (not g.is_empty and g.geom_type == "LineString")]
        )

centrality_segments = gpd.GeoDataFrame(
    {"topo_id": np.arange(len(topological_parts), dtype=int)},
    geometry=topological_parts,
    crs=centrality_source.crs,
)

centrality_segments["length_m"] = centrality_segments.length
centrality_segments = centrality_segments[
    centrality_segments["length_m"] > CENTRALITY_MIN_SEGMENT_LENGTH
].copy()
centrality_segments = centrality_segments.reset_index(drop=True)
centrality_segments["topo_id"] = np.arange(len(centrality_segments), dtype=int)
centrality_segments = centrality_segments.set_index("topo_id", drop=False)

print(f"Topological/noded segments: {len(centrality_segments):,}")
print(f"Total noded length: {centrality_segments['length_m'].sum():,.1f} m")
print(f"Median noded segment length: {centrality_segments['length_m'].median():.2f} m")


In [ ]:
# Build graph and calculate edge betweenness centrality

def _node_key(coord):
    return (
        round(float(coord[0]), CENTRALITY_NODE_DECIMALS),
        round(float(coord[1]), CENTRALITY_NODE_DECIMALS),
    )

G = nx.MultiGraph()

for topo_id, geom in centrality_segments.geometry.items():
    coords = list(geom.coords)
    if len(coords) < 2:
        continue

    u = _node_key(coords[0])
    v = _node_key(coords[-1])

    if u == v:
        continue

    G.add_edge(
        u,
        v,
        key=int(topo_id),
        topo_id=int(topo_id),
        length_m=float(geom.length),
    )

print(f"Graph nodes: {G.number_of_nodes():,}")
print(f"Graph edges: {G.number_of_edges():,}")

component_sizes = pd.Series(
    [len(c) for c in nx.connected_components(G)],
    name="n_nodes"
).sort_values(ascending=False)

print(f"Connected components: {len(component_sizes):,}")
print(f"Largest component: {component_sizes.iloc[0]:,} nodes "
      f"({component_sizes.iloc[0] / G.number_of_nodes():.1%} of graph nodes)")
print("Largest 10 components:")
print(component_sizes.head(10).to_string(index=False))

if CENTRALITY_ONLY_LARGEST_COMPONENT:
    largest_nodes = max(nx.connected_components(G), key=len)
    G_for_bc = G.subgraph(largest_nodes).copy()
    print("\nCalculating centrality on largest connected component only.")
else:
    G_for_bc = G
    print("\nCalculating centrality on all connected components.")

bc_kwargs = {
    "normalized": True,
    "weight": "length_m",
}

if CENTRALITY_APPROX_K is not None:
    bc_kwargs["k"] = min(int(CENTRALITY_APPROX_K), G_for_bc.number_of_nodes())
    bc_kwargs["seed"] = CENTRALITY_RANDOM_SEED
    print(f"Using approximate betweenness with k={bc_kwargs['k']:,} sampled nodes.")
else:
    print("Using exact betweenness. This may take a while on large networks.")

edge_bc = nx.edge_betweenness_centrality(G_for_bc, **bc_kwargs)

centrality_segments["betweenness"] = 0.0

for edge_id, value in edge_bc.items():
    # MultiGraph returns (u, v, key), where key is topo_id here.
    if len(edge_id) == 3:
        topo_id = edge_id[2]
        if topo_id in centrality_segments.index:
            centrality_segments.at[topo_id, "betweenness"] = float(value)

print("\nCentrality calculated.")
print(centrality_segments["betweenness"].describe().round(6))
print(f"Non-zero topological edges: {(centrality_segments['betweenness'] > 0).sum():,}")


The resulting graph contains 5,809 nodes and 7,757 edges. Although the graph is divided into 95 connected components, the largest component contains 95.4% of all nodes, indicating that the street network is largely connected after the topological correction.

The remaining components are very small, with the second and third largest containing only 21 and 19 nodes. These are likely to represent peripheral fragments, isolated streets, boundary effects, or minor digitisation gaps rather than major disconnections in the main network.

Betweenness centrality was calculated across all connected components. The resulting distribution is highly skewed, as expected for urban street networks: most segments have low centrality values, while a small number of segments have substantially higher values and therefore act as important connectors within the street system.

The presence of 7,688 non-zero edges out of 7,758 topological segments suggests that the corrected network is suitable for the subsequent spatial analysis.

In [ ]:
# Assign centrality back to the original street_df segments

def _weighted_centrality_transfer(centrality_segments, centrality_source):
    """
    Transfers centrality from the noded topological segments back to the
    original streetscape segments using length-weighted line overlap.
    """
    base = centrality_source[["seg_id", "geometry"]].copy()

    try:
        overlap = gpd.overlay(
            centrality_segments[["topo_id", "betweenness", "length_m", "geometry"]],
            base,
            how="intersection",
            keep_geom_type=False,
        )

        overlap = overlap[
            overlap.geometry.notna()
            & ~overlap.geometry.is_empty
            & overlap.geometry.geom_type.isin(["LineString", "MultiLineString"])
        ].copy()

        overlap["overlap_len"] = overlap.length
        overlap = overlap[overlap["overlap_len"] > CENTRALITY_MIN_SEGMENT_LENGTH / 10].copy()

        if len(overlap) == 0:
            raise ValueError("No line overlaps found between noded centrality segments and street_df.")

        overlap["bc_x_len"] = overlap["betweenness"] * overlap["overlap_len"]

        agg = overlap.groupby("seg_id", as_index=False).agg(
            bc_x_len=("bc_x_len", "sum"),
            overlap_len=("overlap_len", "sum"),
            n_topo_parts=("topo_id", "nunique"),
        )
        agg["betweenness"] = agg["bc_x_len"] / agg["overlap_len"]

        return agg[["seg_id", "betweenness", "overlap_len", "n_topo_parts"]], "length-weighted overlay"

    except Exception as err:
        warnings.warn(
            f"Length-weighted overlay failed ({err}). "
            "Falling back to nearest representative-point transfer."
        )

        pts = centrality_segments[["topo_id", "betweenness", "length_m", "geometry"]].copy()
        pts["geometry"] = pts.geometry.representative_point()

        nearest = gpd.sjoin_nearest(
            pts,
            base,
            how="left",
            max_distance=max(1.0, CENTRALITY_SNAP_TOLERANCE + 1.0),
            distance_col="transfer_dist",
        )

        nearest = nearest.dropna(subset=["seg_id"]).copy()
        nearest["bc_x_len"] = nearest["betweenness"] * nearest["length_m"]

        agg = nearest.groupby("seg_id", as_index=False).agg(
            bc_x_len=("bc_x_len", "sum"),
            overlap_len=("length_m", "sum"),
            n_topo_parts=("topo_id", "nunique"),
        )
        agg["betweenness"] = agg["bc_x_len"] / agg["overlap_len"]

        return agg[["seg_id", "betweenness", "overlap_len", "n_topo_parts"]], "nearest representative-point fallback"


centrality_transfer, transfer_method = _weighted_centrality_transfer(
    centrality_segments,
    centrality_source,
)

# Remove old centrality columns before merging corrected values.
for col in ["betweenness", "betweenness_rank", "betweenness_log", "centrality_overlap_len", "centrality_n_topo_parts"]:
    if col in street_df.columns:
        street_df = street_df.drop(columns=[col])

street_df = street_df.merge(
    centrality_transfer.rename(
        columns={
            "overlap_len": "centrality_overlap_len",
            "n_topo_parts": "centrality_n_topo_parts",
        }
    ),
    on="seg_id",
    how="left",
)

street_df["betweenness"] = street_df["betweenness"].fillna(0.0)
street_df["centrality_overlap_len"] = street_df["centrality_overlap_len"].fillna(0.0)
street_df["centrality_n_topo_parts"] = street_df["centrality_n_topo_parts"].fillna(0).astype(int)

street_df["betweenness_rank"] = street_df["betweenness"].rank(pct=True)
street_df["betweenness_log"] = np.log1p(street_df["betweenness"])

print(f"Transfer method: {transfer_method}")
print("\nCorrected street_df betweenness:")
print(street_df["betweenness"].describe().round(6))
print(f"street_df segments with betweenness > 0: {(street_df['betweenness'] > 0).sum():,} / {len(street_df):,}")

missing_transfer = (street_df["centrality_overlap_len"] == 0).sum()
print(f"Segments without centrality overlap: {missing_transfer:,}")

# Save corrected outputs to the cache GeoPackage, if DATA_DIR exists.
try:
    cache_path = DATA_DIR / "helwan_cache.gpkg"
    centrality_segments.to_file(cache_path, layer="centrality_topological_segments", driver="GPKG")
    street_df.to_file(cache_path, layer="streetscape_with_corrected_centrality", driver="GPKG")
    print(f"\nSaved corrected centrality layers to: {cache_path}")
    print("Layers: centrality_topological_segments, streetscape_with_corrected_centrality")
except Exception as err:
    print(f"\nCould not save corrected centrality layers automatically: {err}")


In [ ]:
# Quick visual check of corrected centrality

vmax = street_df["betweenness"].quantile(0.95)
if not np.isfinite(vmax) or vmax <= 0:
    vmax = None

street_df.to_crs(epsg=4326).explore(
    column="betweenness",
    cmap="plasma",
    tiles=None,
    tooltip=["betweenness", "betweenness_rank", "seg_id", "centrality_n_topo_parts"],
    style_kwds={"weight": 2, "opacity": 0.85},
    vmax=vmax,
    legend=True
)


The map shows the spatial distribution of edge betweenness centrality across the corrected topological street network. Most street segments display low values, while a limited number of main connectors have substantially higher centrality. This is a skewed pattern because betweenness centrality tends to concentrate on streets that mediate movement between different parts of the network.

For visualisation purposes, the colour scale is limited to the lower range of the distribution. This prevents a small number of extremely high values from dominating the map and allows differences among the majority of street segments to remain visible. Therefore, the upper limit shown in the legend should be interpreted as a cartographic threshold rather than the absolute maximum value of the centrality distribution.

## sDNA betweenness centrality analysis (The correct one)

The betweenness centrality calculated in the previous section treats all street segments equally: it counts how many shortest routes pass through each segment, measuring distance in metres. This works well for understanding global network structure, but it has a limitation since it does not account for how people actually navigate. When walking, people tend to follow straight, legible routes and avoid frequent turns, even if a more direct path exists. 
So, angular betweenness centrality is considered a better way to capture people movement instead of a metric and global centrality analysis.
In order to do this, we need to use sDNA plugin in QGIS. sDNA addresses this by measuring angular betweenness: instead of counting the shortest routes by distance, it finds the routes that require the least change in direction. A straight street has low angular cost; a winding or frequently turning street has high angular cost. This better reflects how pedestrians experience and choose routes through a city.
The analysis is run at three radii (400m, 800m, and 1200m) rather than across the entire network. 
Why is this important?
This matters because ground floor economic activity tends to respond to local foot traffic, not to city-wide movement patterns. A shop depends on people passing within walking distance, not on its position in the network as a whole.

In [ ]:
sdna = gpd.read_file(r"C:\Users\alexa\Urban morphometrics\notebooks\Street-Building Interface\Data\Helwan\Roads_Helwan_sDNA_final.gpkg").to_crs(CRS_PROJ)
print(sdna.columns.tolist())
print(f"Segmentos sDNA: {len(sdna)}")

We can see all the columns that sDNA gave us.
But we only need 4 of them: NQPDA400, NQPDA800, NQPDA1200, NQPDAn
We will join them to street_df, which already has the morphologic data of os, sb, HW, built_coverage, the clusters and the POIs.
By joining the sDNA data of betweenness centrality analysis we can later compare all this variables

In [ ]:
sdna_cols = ["NQPDA400", "NQPDA800", "NQPDA1200", "NQPDAn", "geometry"]
sdna_slim = sdna[sdna_cols].copy()

joined = gpd.sjoin_nearest(
    street_df[["seg_id", "geometry"]],
    sdna_slim,
    how="left",
    distance_col="dist_sdna"
)

for col in ["NQPDA400", "NQPDA800", "NQPDA1200", "NQPDAn"]:
    if col in street_df.columns:
        street_df = street_df.drop(columns=[col])

street_df = street_df.merge(
    joined[["seg_id", "NQPDA400", "NQPDA800", "NQPDA1200", "NQPDAn"]].drop_duplicates("seg_id"),
    on="seg_id", how="left"
)

print(street_df[["NQPDA400", "NQPDA800", "NQPDA1200", "NQPDAn"]].describe().round(3))

The values look interesting, but lets plot the 3 maps of betweenness centrality analysis of the 3 radii (400m, 800m and 1200m):

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import mapclassify

fig, axes = plt.subplots(1, 4, figsize=(26, 7))

titles = ["NQPDA 400m", "NQPDA 800m", "NQPDA 1200m", "NQPDA Global"]
cols   = ["NQPDA400",   "NQPDA800",   "NQPDA1200",   "NQPDAn"]

for ax, col, title in zip(axes, cols, titles):
    classifier = mapclassify.JenksCaspall(street_df[col].dropna(), k=5)
    street_df["jenks_class"] = classifier.find_bin(street_df[col].fillna(0))
    
    buildings_study.plot(ax=ax, color="#e8e4dc", edgecolor="#ccc", linewidth=0.2)
    street_df.plot(
        column="jenks_class", ax=ax,
        cmap="Spectral_r",
        linewidth=1.8,
        legend=False,
    )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_axis_off()

fig.suptitle(
    "Helwan — Angular betweenness centrality (sDNA)",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig("helwan_sdna_nqpda.png", dpi=150, bbox_inches="tight")
plt.show()

The maps above show the distribution of angular betweenness centrality (NQPDA) across the street network of Helwan at three radii: 400m, 800m, and 1200m. The analysis was performed using sDNA (Spatial Design Network Analysis).
At 400m, centrality is highly concentrated along a few short segments in the historical core, reflecting the local navigational importance of the main commercial streets within a short walking distance. At 800m and 1200m, the pattern expands progressively: the main corridor running through the centre of Helwan becomes clearly dominant, and secondary axes in the southern fabric begin to emerge.
This analysis replaces the metric edge betweenness centrality calculated earlier using NetworkX. The NetworkX approach measures how many shortest paths (weighted by segment length) pass through each segment across the entire network. While methodologically valid, it treats all turns as costless and is sensitive to how the network is segmented, producing results that showed almost no difference between high and lower centrality segments in relation to economic activity.
The sDNA angular betweenness addresses both limitations. It calculates the least-angular-change routes rather than the shortest metric routes, penalising directional changes and better reflecting how pedestrians navigate. It also operates at defined radii rather than globally, capturing local movement potential rather than abstract network position. The results confirmed that angular centrality at 400m–800m is considerably more discriminating than global metric betweenness in identifying where economic activity is located in Helwan.

## **POIs of Helwan**

Now its time to check the Points of Interest of Helwan. For Prague and Stockholm we used OSMnx data on amneties. For Helwan we will use the mix-used buildinds and non-residential buildings.

In [ ]:
# Proxy of economical activity: non_residential and mix_use buildings
economic_activity = buildings_study[
    buildings_study["category"].isin(["non_residential", "mix_use"])
].copy()
economic_activity["geometry"] = economic_activity.geometry.centroid
economic_activity = gpd.GeoDataFrame(economic_activity, crs=CRS_PROJ)

print(f"Total POIs proxy: {len(economic_activity)}")
print(economic_activity["category"].value_counts())

Lets plot them in a map to check:

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

# Residential buildings in background
buildings_study[buildings_study["category"] == "residential"].plot(
    ax=ax, color="#e8e4dc", edgecolor="#ccc", linewidth=0.1
)

# Streets
streets_study.plot(ax=ax, color="#999", linewidth=0.3, alpha=0.5)

# Economical activity (mix use and non-residential)
colors_eco = {"mix_use": "#fdae61", "non_residential": "#d7191c"}
for cat, color in colors_eco.items():
    economic_activity[economic_activity["category"] == cat].plot(
        ax=ax, color=color, markersize=3, alpha=0.7, label=cat
    )

ax.legend(fontsize=9)
ax.set_title("Helwan — Economic Activity Distribution\n(non-residential + mix-use buildings)",
             fontsize=12, fontweight="bold")
ax.set_axis_off()
plt.tight_layout()
plt.savefig("helwan_economic_activity.png", dpi=150, bbox_inches="tight")
plt.show()

We can make some interesting readings from this map:
1. The distribution of economical activity is clearly heterogeneous. Theres a clear concentration in the old part of the town (left) and a dispersion on the new part (right)
2. Mix use buildinds concentrate where the urban fabric is more dense. Non residential is clearly in periphery. Regardless there are still mix use (orange) in the periphery and non residential (red) in the old part. Those are "outliers" that probably are more interesting to investigate (Why are those activities in those areas?)

Now we need to do the nearest_segment process: for each building with economic activity (mix use and non_residential) in Helwan, will search for its nearest segment and "give" its attribute to the street segment. This way every street segment will have the count of how many buildings "have economic activity to it".
So lets do the nearest segment:

In [ ]:
pois_seg = gpd.sjoin_nearest(
    economic_activity[["geometry","category"]],
    street_df[["geometry","seg_id"]],
    how="left",
    distance_col="dist_to_street"
)

for cat in ["mix_use","non_residential"]:
    counts = pois_seg[pois_seg["category"] == cat].groupby("seg_id").size()
    counts = counts.reset_index(name=f"n_{cat}")
    street_df = street_df.merge(counts, on="seg_id", how="left")
    street_df[f"n_{cat}"] = street_df[f"n_{cat}"].fillna(0)

street_df["n_pois_total"] = street_df["n_mix_use"] + street_df["n_non_residential"]

print(street_df[["n_mix_use","n_non_residential","n_pois_total"]].describe().round(2))
print(f"\nSegmentos com actividade: {(street_df['n_pois_total'] > 0).sum()}")

This data is also interesting. 
There are 2706 segment with eco_activity in a total of 7236. Thats about 37%

Each variable represents the number of buildings of each category assigned 
to each street segment via nearest neighbour join:

- **n_mix_use** : number of mix-use buildings (ground floor commercial + residential above)
- **n_non_residential** : number of non-residential buildings (services, workshops, equipment)
- **n_pois_total** : total economic activity proxy (sum of the two categories above)
- 


**Reading the percentiles**

- **0% (min)** : the lowest value. THe result is 0, meaning some segments have no activity at all.
- **25%** : 25% of segments have this value or less. The result is still 0, meaning at least 1 in 4 segments has zero activity.
- **50% (median)** : half the segments are below this value, half above. Is 0 again, meaning that more than half of all segments have zero economic activity.
- **75%** : 75% of segments have this value or less. Here we have 1, so means that 75% of segments have 0 or 1 economic activity building nearby.
- **100% (max)** : the highest value. Here we have 33, meaning that one segment has 33 buildings of economic activity assigned to it.

**Interpretation**

The distribution is highly skewed because most street segments are purely residential 
and a few key corridors concentrate all economic activity. This is a typical 
pattern of informally grown cities where commerce emerges along specific axes 
rather than being distributed across the urban fabric.

Only **28% of segments** (5,806 out of 20,454) have at least one economic 
activity building assigned. This might day that we have a concentrated, corridor-based 
commercial structure of Helwan.

## **Morphologycal Clustering**

## Morphological Street Typology — K-Means Clustering

The goal of this step is to identify distinct morphological street types 
based on four streetscape descriptors derived from momepy streetscape:

- **os** : Open Space: visible street width from the sight point (m)
- **built_coverage** : Coverage Ratio: proportion of street frontage occupied by building façades (0–1)
- **sb** : Setback: mean building setback from the street edge (m)
- **HW** : Height/Width ratio: building height relative to street width (enclosure)

These four variables capture the key dimensions of the street-building interface. Meaning  
how wide the street is, how continuously buildings face it, how far they are set back, 
and how enclosed the street feels.

### Method

K-Means clustering with spatial lag is applied to group street segments into 
morphologically similar types. The spatial lag incorporates the neighbourhood 
context of each segment — ensuring that clusters are spatially coherent rather 
than isolated, producing recognisable street character areas rather than 
scattered individual segments.

Variables are standardised before clustering (StandardScaler) to ensure 
that no single variable dominates due to differences in scale.

The optimal number of clusters is determined using the **elbow method** — 
plotting inertia against the number of clusters and identifying the point 
where additional clusters produce diminishing returns.

In [ ]:
from sklearn import cluster, preprocessing

vars_morph = ["os", "built_coverage", "sb", "HW"]

scaler = preprocessing.StandardScaler()
X = scaler.fit_transform(
    street_df[vars_morph].fillna(street_df[vars_morph].median())
)

inertias = {}
for k in range(2, 10):
    km = cluster.KMeans(n_clusters=k, random_state=42)
    km.fit(X)
    inertias[k] = km.inertia_

pd.Series(inertias).plot(title="Elbow method — Helwan")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.tight_layout()
plt.show()

Elbow Method helps us choosing the number of clusters

The elbow plot suggests that the most appropriate number of clusters is around 5. The reduction in inertia is strong between 2 and 5 clusters, but after this point the curve becomes flatter, indicating diminishing returns from adding more clusters.

Although higher values such as 7 clusters still reduce inertia, the improvement is less clearly associated with a major structural change in the curve. Therefore, 5 clusters are selected as a parsimonious solution, balancing model simplicity and differentiation between street types.

In [ ]:
from libpysal import graph, weights

N_CLUSTERS = 5

try:
    queen = graph.Graph.build_contiguity(street_df)
except:
    w = weights.Queen.from_dataframe(street_df, silence_warnings=True)
    queen = graph.Graph.from_W(w)

queen_r = queen.transform("R")
df_std = pd.DataFrame(X, columns=vars_morph)
for col in vars_morph:
    street_df[col + "_lag"] = queen_r.lag(df_std[col].values)

vars_spatial = vars_morph + [v + "_lag" for v in vars_morph]
X_spatial = street_df[vars_spatial].fillna(0).values

km_lag = cluster.KMeans(n_clusters=N_CLUSTERS, random_state=42)
km_lag.fit(X_spatial)
street_df["cluster"] = km_lag.labels_
street_df["cluster_label"] = street_df["cluster"].map(
    {k: f"Type {k+1}" for k in range(N_CLUSTERS)}
)

print(street_df["cluster"].value_counts().sort_index())

We can see that theres a good distribution because theres a consisnt number of segments in each cluster.
Now lets plot the 5 clusters in a map:

In [ ]:
colors_5 = ["#e41a1c","#377eb8","#4daf4a","#ff7f00","#984ea3"]

street_df.to_crs(epsg=4326).explore(
    column="cluster_label",
    categorical=True,
    cmap=colors_5,
    tiles=None,
    tooltip=["cluster_label","os","built_coverage","sb","HW","n_pois_total","betweenness"],
    style_kwds={"weight": 2, "opacity": 0.8},
    legend=True,
    name="Morphological typology"
)

Lets check the morphologycal profile of each cluster:

In [ ]:
print(street_df.groupby("cluster_label")[vars_morph].mean().round(2))




**Morphological Street Types**

The cluster profiles reveal a clear morphological gradient from compact and enclosed streets to wide and open road environments.

Type 1 represents the most compact street condition, with the lowest openness, the highest built coverage, the narrowest street breadth, and the highest height-to-width ratio. This suggests dense and enclosed street canyons, typical of compact urban fabric.

Type 5 represents the opposite condition. It has very high openness, almost no built coverage, the widest street breadth, and the lowest height-to-width ratio. This type is likely associated with large open roads, peripheral connectors, or infrastructural corridors.

Types 2, 3, and 4 occupy intermediate positions. Type 4 remains relatively compact, Type 2 represents a moderately urban street condition, and Type 3 corresponds to more open and lower density residential or planned street environments.

Overall, the five-cluster solution captures a coherent transition between dense enclosed urban streets and wide open structural roads.


# **Exploratory comparison between centrality, street-building interface type and activity**

Lets plot a map with morphological types and the spatial distribution of economical activities:

In [ ]:
import folium

# Streets with morphological cluster labels
streets_map = street_df.to_crs(epsg=4326).copy()

# Activity buildings
activity = buildings_study[
    buildings_study["category"].isin(["mix_use", "non_residential"])
].copy()

# Convert polygons to representative points
activity["geometry"] = activity.geometry.representative_point()
activity_map = activity.to_crs(epsg=4326)

# Separate activity types
mix_use_map = activity_map[activity_map["category"] == "mix_use"].copy()
non_res_map = activity_map[activity_map["category"] == "non_residential"].copy()

# Cluster colours
cluster_colors = ["#e41a1c", "#377eb8", "#4daf4a", "#ff7f00", "#984ea3"]

# Base map: morphological street types
m = streets_map.explore(
    column="cluster_label",
    categorical=True,
    legend=True,
    cmap=cluster_colors,
    tiles=None,
    linewidth=2.5,
    tooltip=["cluster_label", "os", "built_coverage", "sb", "HW",
         "n_pois_total", "length_m", "betweenness",
         "NQPDA400", "NQPDA800", "NQPDA1200", "NQPDAn"],
    popup=False,
    name="Street-building interface types"
)

# Mixed-use points
mix_use_map.explore(
    m=m,
    color="black",
    marker_type="circle_marker",
    marker_kwds={
        "radius": 0.8,
        "fill": True,
        "fillOpacity": 1
    },
    tooltip=True,
    popup=False,
    name="Mixed-use activity"
)

# Non-residential points
non_res_map.explore(
    m=m,
    color="black",
    marker_type="circle_marker",
    marker_kwds={
        "radius": 0.8,
        "fill": True,
        "fillOpacity": 1
    },
    tooltip=False,
    popup=False,
    name="Non-residential activity"
)

folium.LayerControl(collapsed=False).add_to(m)

m

### Exploratory comparison: activity, centrality and street-building interface types

The previous map provided a visual comparison between the morphological street types and the spatial distribution of economic activity. It suggests that activity is not evenly distributed across the street network, nor is it confined to a single type of street environment.

The following section moves to a simple descriptive comparison. Streets are grouped according to their network centrality and street-building interface type. The aim is to examine whether activity is more frequent in certain centrality classes, certain morphological types, or specific combinations of both.

This approach is exploratory. It is intended to identify spatial patterns and formulate research questions, rather than to establish causal relationships.

## How is the economical activity distributed in relation to network centrality and street-building interface types?

This section explores whether street level economic activity is associated with network centrality, street-building interface types, or the combination of both. 

The comparison is organised around three questions:

1. Are highly central streets more likely to contain activity?
2. Does activity vary across street-building interface types?
3. Does centrality matter differently depending on the street-building interface type?

In [ ]:
# Exploratory comparison: activity, centrality, streetscape
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Segment length
street_df["length_m"] = street_df.geometry.length

# Activity variables
street_df["has_activity"] = (street_df["n_pois_total"] > 0).astype(int)


# Angular betweenness classes (sDNA)
q75_400 = street_df["NQPDA400"].quantile(0.75)
street_df["centrality_class_400"] = np.where(
    street_df["NQPDA400"] >= q75_400,
    "High centrality",
    "Lower centrality"
)

q75_800 = street_df["NQPDA800"].quantile(0.75)
street_df["centrality_class_800"] = np.where(
    street_df["NQPDA800"] >= q75_800,
    "High centrality",
    "Lower centrality"
)

q75_1200 = street_df["NQPDA1200"].quantile(0.75)
street_df["centrality_class_1200"] = np.where(
    street_df["NQPDA1200"] >= q75_1200,
    "High centrality",
    "Lower centrality"
)

# Check
street_df[
    [
        "n_pois_total", "has_activity", "length_m",
        "NQPDA400", "centrality_class_400",
        "NQPDA800", "centrality_class_800",
        "NQPDA1200", "centrality_class_1200",
        "cluster_label"
    ]
].head()

This table is a preview of the working dataset. Each row represents one street segment.
n_pois_total counts the number of activity-related buildings associated with each segment. has_activity simplifies this into a binary indicator: 1 when at least one activity is present, 0 when none is recorded. length_m records the segment length in metres.
NQPDA400, NQPDA800, and NQPDA1200 measure normalised angular betweenness centrality at radii of 400m, 800m, and 1200m, calculated using sDNA. Each has a corresponding centrality class: High centrality for segments in the top 25% of the distribution, Lower centrality for the remaining 75%.
cluster_label identifies the street-building interface type assigned to each segment.

### Activity by centrality

Are highly central streets more likely to contain activity?

In [ ]:
results = {}

for centrality_col, class_col, label in [
    ("NQPDA400",     "centrality_class_400",  "Angular betweenness 400m (sDNA)"),
    ("NQPDA800",     "centrality_class_800",  "Angular betweenness 800m (sDNA)"),
    ("NQPDA1200",    "centrality_class_1200", "Angular betweenness 1200m (sDNA)"),
]:
    agg = (
        street_df
        .groupby(class_col, observed=True)
        .agg(
            n_segments=("has_activity", "count"),
            segments_with_activity=("has_activity", "sum"),
            activity_rate=("has_activity", "mean"),
            mean_pois=("n_pois_total", "mean"),
            median_pois=("n_pois_total", "median"),
            mean_length_m=("length_m", "mean"),
            mean_centrality=(centrality_col, "mean"),
        )
    )
    agg["activity_rate_%"] = agg["activity_rate"] * 100
    results[label] = agg.drop(columns="activity_rate").round(2)
    print(f"\n=== {label} ===")
    print(results[label])

This table compares activity between lower-centrality and high-centrality street segments. High-centrality segments correspond to the top 25% of the betweenness distribution, while lower-centrality segments correspond to the remaining 75%.

The results show that both groups have a very similar share of segments with activity: around 40.6% for high-centrality streets and 40.8% for lower-centrality streets. The average number of POIs per segment is also very similar between the two groups. 

This suggests that angular centrality at short radii (400m–800m) is more clearly associated with the presence of activity than at larger radii. Streets that are more accessible to pedestrians within a short walking distance are more likely to contain active ground floor uses. As the radius increases, the discriminating power of centrality decreases — supporting the idea that economic activity at street level responds primarily to local foot traffic rather than to city-wide network position.

### Activity by street-building type:

Does activity vary across street-building interface types?

In [ ]:
activity_by_type = (
    street_df
    .groupby("cluster_label")
    .agg(
        n_segments=("has_activity", "count"),
        segments_with_activity=("has_activity", "sum"),
        activity_rate=("has_activity", "mean"),
        mean_pois=("n_pois_total", "mean"),
        median_pois=("n_pois_total", "median"),
        mean_length_m=("length_m", "mean"),
        mean_betweenness=("betweenness", "mean"),
        mean_NQPDA400=("NQPDA400", "mean"),
        mean_NQPDA800=("NQPDA800", "mean"),
        mean_NQPDA1200=("NQPDA1200", "mean"),
        os=("os", "mean"),
        built_coverage=("built_coverage", "mean"),
        sb=("sb", "mean"),
        HW=("HW", "mean"),
    )
)
activity_by_type["activity_rate_%"] = activity_by_type["activity_rate"] * 100
activity_by_type.drop(columns="activity_rate").round(2)

The table compares activity levels across the five street building interface types. Its telling us the number of street segments, the number of segments with at least one activity, the average number of POIs, and the percentage of active segments for each type. We are basically analysing the data from the perspetive of the types.

Activity varies clearly across street-building interface types.
Type 3 and Type 1 are the most active, with 51.4% and 49.2% of segments containing at least one activity. These are compact, enclosed street environments with high built coverage (0.47 and 0.66), short setbacks (10m and 6.6m), and high height-to-width ratios (2.84 and 4.54). They also show the highest angular centrality values at all radii.
Type 2 is the least active at 2.5%, characterised by very high openness (os=99m), near-zero built coverage (0.01), and very large setbacks (48m). This type corresponds to infrastructure corridors or open roads with no meaningful street-building interface. Its angular centrality values are also the lowest across all radii.
Type 5 shows moderate activity (25.3%) despite having the highest metric betweenness. Its openness (os=77m) and low built coverage (0.16) suggest a wide, discontinuous street environment where centrality alone does not generate activity.
Overall, the results indicate that the morphological characteristics of the street-building interface particularly built coverage, setback, and enclosure are more consistently associated with activity than centrality measures alone.

### Activity by centrality × interface type

Does centrality matter differently depending on the street-building interface type?

In [ ]:
for class_col, label in [
    ("centrality_class_400",  "Angular betweenness 400m (sDNA)"),
    ("centrality_class_800",  "Angular betweenness 800m (sDNA)"),
    ("centrality_class_1200", "Angular betweenness 1200m (sDNA)"),
]:
    matrix = (
        street_df
        .pivot_table(
            index="cluster_label",
            columns=class_col,
            values="has_activity",
            aggfunc="mean",
            observed=True,
        ) * 100
    )
    print(f"\n=== {label} ===")
    print(matrix.round(1))

The table shows activity rates for each combination of street-building interface type and centrality class, at three radii.
For Type 1 and Type 4, centrality has a small but consistent positive effect at 400m and 800m with highly central segments being slightly more active. At 1200m this effect disappears or reverses, suggesting that for these compact types, local pedestrian accessibility matters more than wider network position.
Type 3 shows a consistent reversal: at 800m and 1200m, lower-centrality segments are more active than high-centrality ones (52.3% vs 49.1% at 800m; 54.5% vs 43.8% at 1200m). This suggests that activity in this type is distributed across the fabric rather than concentrated on the most central streets.
Type 5 reverses at 400m with lower-centrality segments being more active (26.6% vs 17.1%). This is the most open and discontinuous type, where high angular centrality at short radii likely corresponds to major traffic arteries with little pedestrian-oriented activity.
Type 2 remains consistently inactive regardless of centrality or radius, confirming that below a certain morphological threshold, centrality has no meaningful effect.
Overall, the results show that centrality and interface type interact differently depending on the morphological context. Centrality has the clearest positive effect in compact, enclosed types at local radii. In more open or discontinuous environments, the effect is weak, absent, or reversed.

In [ ]:
streets_clean.to_file("helwan_sdna.gpkg", layer="streets", driver="GPKG")
print("Exportado.")